# Car Sales Environment Prompt Demo

This notebook walks through a short manual car-sales exchange and shows the exact prompts at each step.

It focuses on:
- the initial buyer question prompt
- the seller response prompt
- the next buyer prompt after a seller reply


In [ ]:
from pathlib import Path
from pprint import pprint
from types import SimpleNamespace
import importlib
import sys

NOTEBOOK_ROOT = Path.cwd().resolve()
REPO_ROOT = next((candidate for candidate in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents] if (candidate / 'Environments').exists() and (candidate / 'LocalizationScripts').exists()), NOTEBOOK_ROOT)
ENV_SRC = REPO_ROOT / 'Environments' / 'CarSales' / 'src'
if str(ENV_SRC) not in sys.path:
    sys.path.insert(0, str(ENV_SRC))

import car_sales_environment as car_sales_env
importlib.reload(car_sales_env)

UsedCarSalesEnvironment = car_sales_env.UsedCarSalesEnvironment
CarSalesSpec = car_sales_env.CarSalesSpec
print('Imported from:', ENV_SRC / 'car_sales_environment.py')


In [ ]:
def make_agents():
    return [
        SimpleNamespace(name='Seller', reasoning_instruction='COD', instruction_format='reasoning'),
        SimpleNamespace(name='Buyer', reasoning_instruction='COD', instruction_format='reasoning'),
    ]


def make_env(seed=0, scenario_name='ford_f150_xlt', max_rounds=4):
    return UsedCarSalesEnvironment(
        agents=make_agents(),
        seed=seed,
        scenario_name=scenario_name,
        spec=CarSalesSpec(max_rounds=max_rounds),
        history_window=12,
    )


def show_messages(messages):
    for idx, msg in enumerate(messages):
        print(f'===== MESSAGE {idx} / role={msg["role"]} =====')
        print(msg['content'])
        print()


def show_dialogue(dialogue_history):
    for item in dialogue_history:
        extras = []
        if item.get('question_focus'):
            extras.append(f"focus={item['question_focus']}")
        if item.get('disclosed_issue_ids'):
            extras.append(f"disclosed={item['disclosed_issue_ids']}")
        suffix = '' if not extras else ' [' + '; '.join(extras) + ']'
        print(f"{item['speaker']}: {item['message']}{suffix}")
        print()


In [ ]:
env = make_env(seed=0, scenario_name='ford_f150_xlt')
state = env.get_state()

print('phase:', state['phase'])
print('active_player:', state['active_player'])
print('scenario:', state['scenario']['name'])
print('current_round:')
pprint(state['current_round'])
print('\nhidden known issue ids:', state['hidden_state']['seller_known_issue_ids'])
print('\npublic dialogue history:')
show_dialogue(state['dialogue_history'])
print('\ninitial prompt:')
show_messages(state['messages'])


In [ ]:
buyer_action = env.get_truthful_action()
buyer_result = env.manual_step(buyer_action)

print('truthful buyer action:')
pprint(buyer_action)
print('\nbuyer result:')
pprint(buyer_result)
print('\nnext prompt (seller response):')
show_messages(env.get_state()['messages'])


In [ ]:
seller_action = env.get_truthful_action()
seller_result = env.manual_step(seller_action)

print('truthful seller action:')
pprint(seller_action)
print('\ntruthful seller label:')
pprint(seller_result['label'])
print('\ntruthful seller resolution:')
pprint(seller_result['resolution'])
print('\nnext prompt (buyer question):')
show_messages(env.get_state()['messages'])
